In [1]:
import pandas as pd 
import numpy as np 
import math as math
import joblib
from joblib import dump
import os

In [2]:
import pandas as pd 
print(np.__version__)

1.24.4


## LOAD DATA

In [2]:
def load_Dataset(baseFile, fold_Number): 
    rnmColData = ['user_id', 'item_id', 'rating', 'timestamp']
    base_File = f"{baseFile}/u{fold_Number}.base"
    test_File = f"{baseFile}/u{fold_Number}.test"
    base_Data = pd.read_csv(base_File, sep="\t", header=None, names=rnmColData)
    test_Data = pd.read_csv(test_File, sep="\t", header=None, names=rnmColData)
    base_Data = base_Data.drop(columns=["timestamp"])
    test_Data = test_Data.drop(columns=["timestamp"])
    return base_Data, test_Data

In [5]:
call_base = "ml-100k"
basedata, testdata = load_Dataset(call_base, 4)
basedata

,user_id,item_id,rating
0,1,1,5
1,1,2,3
2,1,3,4
3,1,5,3
4,1,6,5
...,...,...,...
79995,943,1028,2
79996,943,1044,3
79997,943,1047,2
79998,943,1228,3


## CONVERT RATING MATRIKS

In [3]:
def ConvertRatingMatriks(ratingData):
    # membuat container untuk rating matriks dengan ukuran user x item
    matriks_rating = pd.DataFrame(np.zeros((943, 1682)), columns = list(range(1, 1683)), index = list(range(1, 944)))
    # merubah data frame ke dalam bentuk matriks rating pivot
    convertMatriksRating = ratingData.pivot_table(index = 'user_id', columns = 'item_id', values='rating')
    # mengisi matriks rating NaN dengan 0
    convertMatriksRating = convertMatriksRating.fillna(0)
    # updating matriks rating dengan matriks rating yang sudah di pivot
    matriks_rating.update(convertMatriksRating)
    # mengembalikan matriks rating
    return matriks_rating

## BASE MODEL

In [6]:
rating_Matriks = ConvertRatingMatriks(basedata)
rating_Matriks

,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
1,5.0,3.0,4.0,0.0,3.0,5.0,0.0,1.0,5.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,4.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
939,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
940,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
941,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
942,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### MEAN 

In [12]:
def meanRating(RatingMatriks, jenis="user-based"):
    if jenis == "user-based":
        # menghitung berapa banyak rating yang ada pada setiap user(baris)
        axis = 1
        # index baris user dari rating matriks
        index = RatingMatriks.index
    elif jenis == "item-based":
        # menghitung berapa banyak rating yang ada pada setiap item(kolom)
        axis = 0
        # index kolom item dari rating matriks
        index = RatingMatriks.columns
    # menghitung mean rating dari setiap user/item
    pembilang = RatingMatriks.sum(axis=axis)
    # menghitung banyaknya rating yang ada pada setiap user/item yang bukan 0
    penyebut = np.count_nonzero(RatingMatriks, axis=axis)
    # cek jika penyebut == 0, maka mean rating = 0, jika tidak hitung mean rating
    calculateMeanRating = np.where(penyebut == 0, 0, pembilang / penyebut)
    # mengembalikan mean rating dalam bentuk data frame
    meanRatingNew = pd.DataFrame(calculateMeanRating, index=index, columns=["meanRating"])
    return meanRatingNew

#### USER

In [55]:
cal_MeanRatingUser = meanRating(rating_Matriks, jenis="user-based")
cal_MeanRatingUser

,meanRating
1,3.597610
2,3.719298
3,2.788462
4,4.350000
5,2.864198
...,...
939,4.424242
940,3.484848
941,3.866667
942,4.319149


#### ITEM

In [56]:
cal_MeanRatingItem = meanRating(rating_Matriks, jenis="item-based")
cal_MeanRatingItem

,meanRating
1,3.891967
2,3.168317
3,3.041667
4,3.612121
5,3.352113
...,...
1678,0.000000
1679,0.000000
1680,2.000000
1681,3.000000


### MEAN-CENTERED

In [15]:
def meanCenteredRating(RatingMatriks, meanRating, jenis="user-based"):
    # matriks rating jadi numpy array
    npRatingMatriks = np.array(RatingMatriks)
    if jenis == "user-based":
        # Reshape meanRating menjadi (jumlah_user, 1), kolom
        npMeanRating = np.array(meanRating).reshape(-1, 1)  # (943, 1)
    elif jenis == "item-based":
        # Reshape meanRating menjadi (1, jumlah_item), baris
        npMeanRating = np.array(meanRating).reshape(1, -1)  # (1, 1650)
    # Menghitung mean centered, hanya mengurangi rating yang bukan 0, jika rating 0 maka hasilnya 0
    meanCentered = np.where(npRatingMatriks != 0, npRatingMatriks - npMeanRating, 0)
    # Mengembalikan dalam bentuk DataFrame
    dfMeanCentered = pd.DataFrame(meanCentered, index=RatingMatriks.index, columns=RatingMatriks.columns)
    return dfMeanCentered

#### USER

In [57]:
cal_MeanCenteredUser = meanCenteredRating(rating_Matriks, cal_MeanRatingUser, jenis="user-based")
cal_MeanCenteredUser

,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
1,1.402390,-0.597610,0.40239,0.0,-0.59761,1.40239,0.000000,-2.59761,1.402390,-0.597610,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.280702,0.000000,0.00000,0.0,0.00000,0.00000,0.000000,0.00000,0.000000,-1.719298,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.000000,0.000000,0.00000,0.0,0.00000,0.00000,0.000000,0.00000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.000000,0.000000,0.00000,0.0,0.00000,0.00000,0.000000,0.00000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,1.135802,0.135802,0.00000,0.0,0.00000,0.00000,0.000000,0.00000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
939,0.000000,0.000000,0.00000,0.0,0.00000,0.00000,0.000000,0.00000,0.575758,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
940,0.000000,0.000000,0.00000,0.0,0.00000,0.00000,0.515152,0.00000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
941,1.133333,0.000000,0.00000,0.0,0.00000,0.00000,0.000000,0.00000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
942,0.000000,0.000000,0.00000,0.0,0.00000,0.00000,0.000000,0.00000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


#### ITEM

In [58]:
cal_MeanCenteredItem = meanCenteredRating(rating_Matriks, cal_MeanRatingItem, jenis="item-based")
cal_MeanCenteredItem

,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
1,1.108033,-0.168317,0.958333,0.0,-0.352113,1.5,0.000000,-2.969325,1.049587,-0.786667,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.108033,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,-1.786667,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.108033,-0.168317,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
939,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,1.049587,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
940,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.213376,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
941,1.108033,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
942,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### SIMILARITY

In [7]:
def SimilarityRJ(RatingMatriks, index1, index2, jenis="user-based"):
    if jenis == "user-based":
        Rated1 = set(RatingMatriks.columns[RatingMatriks.iloc[index1, :] != 0])
        Rated2 = set(RatingMatriks.columns[RatingMatriks.iloc[index2, :] != 0])
    elif jenis == "item-based":
        Rated1 = set(RatingMatriks.index[RatingMatriks.iloc[:, index1] != 0])
        Rated2 = set(RatingMatriks.index[RatingMatriks.iloc[:, index2] != 0])

    # menghitung jumlah item/user yang sama-sama diberi rating
    intersection = len(Rated1.intersection(Rated2))
    # print("intersection:", intersection)
    # menghitung jumlah item yang hanya dirating oleh satu pihak (un-co-rated)
    NotRated1 = len(Rated1) - intersection
    # print("notRated1:", NotRated1)
    NotRated2 = len(Rated2) - intersection
    # print("notRated2:", NotRated2)
    # menghitung similarity menggunakan rumus Relevant Jaccard
    if intersection != 0:
        rumusRelevantJaccard = 1 / (1 + (1 / intersection) +(NotRated1 / (1 + NotRated1)) +(1 / (1 + NotRated2))
        )
    else:
        rumusRelevantJaccard = 0

    return rumusRelevantJaccard


In [13]:
cal_simRJUser = SimilarityRJ(rating_Matriks, 0, 0, jenis="user-based")
cal_simRJUser

0.4981549815498156

In [14]:
cal_simRJItem = SimilarityRJ(rating_Matriks, 1, 2, jenis="item-based")
cal_simRJItem

0.48889872375808235

In [8]:
def SimilarityAllRJ(RatingMatriks, jenis="user-based"):
    if jenis == "user-based":
        jumlah = RatingMatriks.shape[0]
        simMatriks = np.zeros((jumlah, jumlah))
        for i in range(jumlah):
            for j in range(jumlah):
                nilaiSim = SimilarityRJ(RatingMatriks, i, j, jenis=jenis)
                simMatriks[i][j] = nilaiSim
        simMatriksDf = pd.DataFrame(simMatriks, index=RatingMatriks.index, columns=RatingMatriks.index)
    elif jenis == "item-based":
        jumlah = RatingMatriks.shape[1]
        simMatriks = np.zeros((jumlah, jumlah))
        for i in range(jumlah):
            for j in range(jumlah):
                nilaiSim = SimilarityRJ(RatingMatriks, i, j, jenis=jenis)
                simMatriks[i][j] = nilaiSim
        simMatriksDf = pd.DataFrame(simMatriks, index=RatingMatriks.columns, columns=RatingMatriks.columns)
    return simMatriksDf

In [9]:
calRJALLUser2 = SimilarityAllRJ(rating_Matriks, jenis="user-based")
calRJALLUser2

,1,2,3,4,5,6,7,8,9,10,...,934,935,936,937,938,939,940,941,942,943
1,0.499006,0.481010,0.466609,0.452447,0.495189,0.496445,0.498879,0.482976,0.392367,0.495091,...,0.492395,0.460937,0.489807,0.467729,0.487371,0.469431,0.486307,0.429313,0.477840,0.493015
2,0.490511,0.495652,0.466203,0.451650,0.457151,0.499143,0.489824,0.461197,0.445706,0.487828,...,0.476179,0.464424,0.491935,0.471605,0.488532,0.471513,0.479044,0.444952,0.460565,0.456688
3,0.474636,0.467131,0.495238,0.460818,0.000000,0.477654,0.485021,0.461633,0.394785,0.474161,...,0.430629,0.396217,0.483108,0.440608,0.468383,0.000000,0.467774,0.391877,0.461000,0.334556
4,0.481809,0.471869,0.500848,0.487805,0.407568,0.475036,0.497043,0.471513,0.333333,0.455117,...,0.337987,0.334262,0.454282,0.448194,0.464897,0.000000,0.484255,0.397015,0.462801,0.338036
5,0.497825,0.451969,0.000000,0.392708,0.498462,0.494305,0.499667,0.482528,0.419703,0.491603,...,0.491051,0.394125,0.475236,0.447360,0.482726,0.439166,0.483905,0.429730,0.471747,0.493305
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
939,0.487429,0.480962,0.000000,0.000000,0.449851,0.487171,0.485168,0.334609,0.331159,0.467927,...,0.432915,0.468689,0.483305,0.443131,0.499343,0.492537,0.431648,0.420522,0.334339,0.472922
940,0.496857,0.477480,0.465565,0.457671,0.490498,0.499614,0.502252,0.477029,0.436119,0.497085,...,0.488914,0.395555,0.477735,0.463893,0.476883,0.425538,0.496241,0.391230,0.468812,0.480293
941,0.460681,0.479410,0.408467,0.403030,0.460203,0.490948,0.460951,0.439364,0.403030,0.460251,...,0.410283,0.434430,0.497382,0.453172,0.480685,0.436935,0.409173,0.483871,0.338454,0.410357
942,0.490056,0.462516,0.462079,0.446579,0.480718,0.495103,0.499270,0.471311,0.000000,0.488170,...,0.477369,0.396552,0.464432,0.397567,0.456567,0.332333,0.472378,0.328365,0.494737,0.470110


In [10]:
calRJALLItem2 = SimilarityAllRJ(rating_Matriks, jenis="item-based")
calRJALLItem2

,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
1,0.499308,0.490024,0.484113,0.494775,0.487206,0.456285,0.498144,0.494666,0.496960,0.487782,...,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000
2,0.502842,0.497537,0.486222,0.498007,0.487881,0.328937,0.500601,0.494942,0.496359,0.482727,...,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.250620,0.000000
3,0.507073,0.489398,0.496552,0.495680,0.485638,0.450774,0.507687,0.493496,0.498608,0.481666,...,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.250871
4,0.500179,0.492500,0.487987,0.498489,0.488180,0.434282,0.499401,0.496349,0.497668,0.487207,...,0.0,0.0,0.250379,0.250379,0.0,0.0,0.0,0.000000,0.250379,0.000000
5,0.499617,0.491726,0.485791,0.496973,0.496503,0.329390,0.502335,0.496567,0.499174,0.461723,...,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.250883
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1678,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000
1679,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000
1680,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.333333,0.000000,0.000000
1681,0.000000,0.497537,0.000000,0.498489,0.000000,0.000000,0.499205,0.000000,0.498969,0.000000,...,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.333333,0.000000


In [24]:
print(calRJALLItem2.shape)

(1682, 1682)


### TOP-K

In [59]:
#use .pkl to find topK

cal_RJ = joblib.load("case/sim/RJ/ub/simRJFold4.joblib")
cal_RJ

,1,2,3,4,5,6,7,8,9,10,...,934,935,936,937,938,939,940,941,942,943
1,0.499006,0.481010,0.466609,0.452447,0.495189,0.496445,0.498879,0.482976,0.392367,0.495091,...,0.492395,0.460937,0.489807,0.467729,0.487371,0.469431,0.486307,0.429313,0.477840,0.493015
2,0.481010,0.495652,0.466203,0.451650,0.457151,0.499143,0.489824,0.461197,0.445706,0.487828,...,0.476179,0.464424,0.491935,0.471605,0.488532,0.471513,0.479044,0.444952,0.460565,0.456688
3,0.466609,0.466203,0.495238,0.460818,0.000000,0.477654,0.485021,0.461633,0.394785,0.474161,...,0.430629,0.396217,0.483108,0.440608,0.468383,0.000000,0.467774,0.391877,0.461000,0.334556
4,0.452447,0.451650,0.460818,0.487805,0.407568,0.475036,0.497043,0.471513,0.333333,0.455117,...,0.337987,0.334262,0.454282,0.448194,0.464897,0.000000,0.484255,0.397015,0.462801,0.338036
5,0.495189,0.457151,0.000000,0.407568,0.498462,0.494305,0.499667,0.482528,0.419703,0.491603,...,0.491051,0.394125,0.475236,0.447360,0.482726,0.439166,0.483905,0.429730,0.471747,0.493305
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
939,0.469431,0.471513,0.000000,0.000000,0.439166,0.469671,0.467539,0.332068,0.335536,0.455322,...,0.424315,0.478787,0.469282,0.445765,0.475594,0.492537,0.431648,0.420522,0.334339,0.472922
940,0.486307,0.479044,0.467774,0.484255,0.483905,0.486852,0.485680,0.479499,0.453094,0.487142,...,0.484083,0.404546,0.474655,0.477480,0.475500,0.431648,0.496241,0.391230,0.468812,0.480293
941,0.429313,0.444952,0.391877,0.397015,0.429730,0.444674,0.429079,0.418297,0.397015,0.429687,...,0.390220,0.422868,0.446533,0.436047,0.443859,0.420522,0.391230,0.483871,0.338454,0.410357
942,0.477840,0.460565,0.461000,0.462801,0.471747,0.480718,0.480675,0.469867,0.000000,0.477466,...,0.470056,0.403509,0.458680,0.402463,0.452541,0.334339,0.468812,0.338454,0.494737,0.470110


In [ ]:
# def TopKTetangga(similarity, k):
#     # Mengambil k tetangga terdekat 
#     # Mengurutkan similarity dalam urutan desending
#     return np.argsort(-similarity)[:k]

In [19]:
def TetanggaK(target_user, target_item, rating_matrix_np, similarity_np, k, jenis="user-based"):
    """
    Mengembalikan indeks K tetangga terdekat (user atau item) berdasarkan similarity.
    """
    rating_matrix = rating_matrix_np
    if jenis == "user-based":
        # Ambil semua rating terhadap item target, data rating untuk item target
        item_ratings = rating_matrix[:, target_item]
        tetangga_idx = np.where((item_ratings != 0))[0]
        sim_scores = similarity_np[target_user, tetangga_idx]
        # cek jumalh rating
        martiks = rating_matrix[target_user, tetangga_idx]
        # print(f"matriks rating {martiks.shape}")

    else:  # item-based
        # Di item-based, user tetap user, item adalah target
        user_ratings = rating_matrix_np[target_user, :]
        tetangga_idx = np.where((user_ratings != 0))[0]
        # Ambil skor similarity item-item
        sim_scores = similarity_np[tetangga_idx, target_item]
    # Urutkan tetangga berdasarkan similarity tertinggi
    if len(sim_scores) == 0:
        return []

    # print(f"sim scores {sim_scores}")
    # print(f"sim scores {sim_scores.shape}")
    sorted_idx = np.argsort(-sim_scores)[:k]
    # print(f"sorted idx {sorted_idx}")
    # print(f"sim from soreted {sorted_idx} {sim_scores[sorted_idx]}")
    return tetangga_idx[sorted_idx]



## PREDIKSI

In [ ]:
def PrediksiCF(RatingMatriks, similarityFunction, mean, meanCen, user, item, k=2, jenis="user-based"):
    """
    Menghitung prediksi rating menggunakan Collaborative Filtering berbasis user/item, versi optimal.
    """
    # Cache ke bentuk NumPy
    rating_np = RatingMatriks.to_numpy()
    # cek jumlah yang pernah memberikan rating 
    # print(f"jumlah rating item {item} {np.count_nonzero(rating_np[:, item])}")
    similarity_np = similarityFunction.to_numpy()
    # print(similarity_np.shape)
    mean_np = mean.to_numpy()
    # print(mean_np.shape)
    meanCen_np = meanCen.to_numpy()
    # print("mean cen", meanCen_np.shape)

    # Set indeks untuk pencarian tetangga
    target_user = user
    target_item = item
    # print(f"target user {target_user} {target_item}")
    mean_value = mean_np[user] if jenis == "user-based" else mean_np[item]
    # print(f"mean {mean_value}")

    # Cari tetangga
    tetangga = TetanggaK(target_user, target_item, rating_np, similarity_np, k, jenis)
    # print(f"tetangga {tetangga}")
    if len(tetangga) == 0:
        return float(mean_value)

    # Ambil similarity dan mean centered rating
    if jenis == "user-based":
        sim = similarity_np[user, tetangga] # ambil similarity user-user
        # print(f'sim {sim}')
        mean_cen = meanCen_np[tetangga, item] 
        # print(f"mean cen {mean_cen}")
        # cek tetangga dan item pada mean cen
        # print(f"mean cen {meanCen_np[tetangga, item].shape}")
    else:  # item-based
        sim = similarity_np[tetangga, item] # ambil similarity item-item
        # print(f"sim item {sim.shape}") 
        mean_cen = meanCen_np[user, tetangga] # ambil mean centered rating

    # vektor ((a1 * b1) + (a2 * b2))
    pembilang = np.dot(mean_cen, sim)
    # print(f"pembilang {pembilang}")
    penyebut = np.sum(np.abs(sim))
    # print(f"penyebut {penyebut}")

    prediksi = mean_value + (pembilang / penyebut) if penyebut != 0 else mean_value
    # print(f"prediksi {mean_value} + {pembilang} / {penyebut}")
    return float(prediksi)

In [21]:
def prediksi_semua_matriks2(RatingMatriks, similarityFunction, mean, meanCen, k=2, jenis="user-based"):
    """
    Mengisi semua rating kosong dalam matriks dengan hasil prediksi CF.
    """
    # salin rating matriks
    hasil_matriks = RatingMatriks.copy()
    # mendapatkan ukuran matriks rating/ ambil
    num_users, num_items = hasil_matriks.shape
    # loop setiap user
    for user in range(num_users):
        # loop setiap item
        for item in range(num_items):
            if hasil_matriks.iloc[user, item] == 0:
                pred_rating = PrediksiCF(
                    RatingMatriks=RatingMatriks,
                    similarityFunction=similarityFunction,
                    mean=mean,
                    meanCen=meanCen,
                    user=user,
                    item=item,
                    k=k,
                    jenis=jenis # user-based atau item-based
                )
                # isi prediksi hasil ke dalam matriks
                hasil_matriks.iloc[user, item] = pred_rating
    return hasil_matriks

In [22]:
def simpan_semua_prediksi_k(
    RatingMatriks, similarityFunction, mean, meanCen,
    variasi_k, jenis="user-based",
    folder_output="case/prediksi/Jac/ub/5"
):
    for k in variasi_k:
        # Hitung prediksi
        prediksi = prediksi_semua_matriks2(
            RatingMatriks=RatingMatriks,
            similarityFunction=similarityFunction,
            mean=mean,
            meanCen=meanCen,
            k=k,
            jenis=jenis
        )
        # Siapkan path file
        path_file = os.path.join(folder_output, f"{k}.joblib")
        os.makedirs(folder_output, exist_ok=True)

        # Simpan hasil prediksi
        joblib.dump(prediksi, path_file)
        print(f"✔️ Prediksi untuk k={k} disimpan di: {path_file}")

# LOOP PREDIKSI UB DAN IB

In [1]:
# # loop prediksi user-based
# simpan_semua_prediksi_k(
#     RatingMatriks=rating_Matriks,
#     similarityFunction=cal_RJ,
#     mean=cal_MeanRatingUser,
#     meanCen=cal_MeanCenteredUser,
#     variasi_k=[5, 10, 15, 18, 20, 25, 30, 40, 50, 100, 200],
#     jenis="user-based",
#     folder_output="case/prediksiModel/RJ2/ub/4"
# )

# export to joblib

In [62]:
import joblib
import os

def process_and_save_mean_ratings(baseFile, saveDirectory, jenis="user-based"):
    # Loop untuk 5 fold
    for fold_number in range(1, 6):
        # Load dataset untuk base dan test
        base_Data, _ = load_Dataset(baseFile, fold_number)
        
        # Convert ke rating matriks
        matriks_rating = ConvertRatingMatriks(base_Data)
    
        # simlarity jaccard 
        simRJBase = SimilarityAllRJ(matriks_rating,  jenis=jenis)
        simRJFile = f"{saveDirectory}/simRJFold{fold_number}.joblib"
        joblib.dump(simRJBase, simRJFile)
        print(f"[INFO] sim RJ for fold {fold_number} ({jenis}) saved successfully at {simRJFile}")

# Contoh penggunaan
baseFile = "ml-100k"  # Ganti dengan path file data Anda


saveSimUBrj = "case/sim/RJ/ub"
saveSimIBrj = "case/sim/RJ/ib"

process_and_save_mean_ratings(baseFile, saveSimUBrj, jenis="user-based")

# Menyimpan mean centered untuk item-based
process_and_save_mean_ratings(baseFile, saveSimIBrj, jenis="item-based")

[INFO] sim RJ for fold 1 (user-based) saved successfully at case/sim/RJ/ub/simRJFold1.joblib
[INFO] sim RJ for fold 2 (user-based) saved successfully at case/sim/RJ/ub/simRJFold2.joblib
[INFO] sim RJ for fold 3 (user-based) saved successfully at case/sim/RJ/ub/simRJFold3.joblib
[INFO] sim RJ for fold 4 (user-based) saved successfully at case/sim/RJ/ub/simRJFold4.joblib
[INFO] sim RJ for fold 5 (user-based) saved successfully at case/sim/RJ/ub/simRJFold5.joblib
[INFO] sim RJ for fold 1 (item-based) saved successfully at case/sim/RJ/ib/simRJFold1.joblib
[INFO] sim RJ for fold 2 (item-based) saved successfully at case/sim/RJ/ib/simRJFold2.joblib
[INFO] sim RJ for fold 3 (item-based) saved successfully at case/sim/RJ/ib/simRJFold3.joblib
[INFO] sim RJ for fold 4 (item-based) saved successfully at case/sim/RJ/ib/simRJFold4.joblib
[INFO] sim RJ for fold 5 (item-based) saved successfully at case/sim/RJ/ib/simRJFold5.joblib


In [16]:
dataMatriks = joblib.load("case/ratingMatriks/u1.joblib")
dataMatriks

,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
1,5.0,3.0,4.0,3.0,3.0,0.0,4.0,1.0,5.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
939,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
940,0.0,0.0,0.0,2.0,0.0,0.0,4.0,5.0,3.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
941,5.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
942,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# loop

In [1]:
import pandas as pd
import numpy as np
import math as math
import joblib
from joblib import dump
import os
import time

## user

In [39]:
# mean
ratingMatriks = joblib.load("case/ratingMatriks/u5.joblib")
calSimUser = joblib.load("case/sim/RJ/ub/simRJFold5.joblib")
calMeanUser= joblib.load("case/mean/ub/meanFold5.joblib")
calMeanCenUser = joblib.load("case/meanCen/ub/meanCen5.joblib")

In [11]:
def TetanggaK(target_user, target_item, rating_matrix_np, similarity_np, k, jenis="user-based"):
    """
    Mengembalikan indeks K tetangga terdekat (user atau item) berdasarkan similarity.
    """
    rating_matrix = rating_matrix_np
    if jenis == "user-based":
        # Ambil semua rating terhadap item target, data rating untuk item target
        item_ratings = rating_matrix[:, target_item]
        tetangga_idx = np.where((item_ratings != 0))[0]
        sim_scores = similarity_np[target_user, tetangga_idx]
        # cek jumalh rating
        martiks = rating_matrix[target_user, tetangga_idx]
        # print(f"matriks rating {martiks.shape}")

    else:  # item-based
        # Di item-based, user tetap user, item adalah target
        user_ratings = rating_matrix_np[target_user, :]
        tetangga_idx = np.where((user_ratings != 0))[0]
        # Ambil skor similarity item-item
        sim_scores = similarity_np[tetangga_idx, target_item]
    # Urutkan tetangga berdasarkan similarity tertinggi
    if len(sim_scores) == 0:
        return []

    # print(f"sim scores {sim_scores}")
    # print(f"sim scores {sim_scores.shape}")
    sorted_idx = np.argsort(-sim_scores)[:k]
    # print(f"sorted idx {sorted_idx}")
    # print(f"sim from soreted {sorted_idx} {sim_scores[sorted_idx]}")
    return tetangga_idx[sorted_idx]



In [12]:
def PrediksiCF(RatingMatriks, similarityFunction, mean, meanCen, user, item, k=2, jenis="user-based"):
    """
    Menghitung prediksi rating menggunakan Collaborative Filtering berbasis user/item, versi optimal.
    """

    # Cache ke bentuk NumPy
    rating_np = RatingMatriks.to_numpy()

    # cek jumlah yang pernah memberikan rating 
    # print(f"jumlah rating item {item} {np.count_nonzero(rating_np[:, item])}")


    similarity_np = similarityFunction.to_numpy()
    # print(similarity_np.shape)
    mean_np = mean.to_numpy()
    # print(mean_np.shape)
    meanCen_np = meanCen.to_numpy()
    # print("mean cen", meanCen_np.shape)

    # Set indeks untuk pencarian tetangga
    target_user = user
    target_item = item
    # print(f"target user {target_user} {target_item}")
    mean_value = mean_np[user] if jenis == "user-based" else mean_np[item]
    # print(f"mean {mean_value}")

    # Cari tetangga
    tetangga = TetanggaK(target_user, target_item, rating_np, similarity_np, k, jenis)
    # print(f"tetangga {tetangga}")
    if len(tetangga) == 0:
        return float(mean_value)

    # Ambil similarity dan mean centered rating
    if jenis == "user-based":
        sim = similarity_np[user, tetangga] # ambil similarity user-user
        # print(f'sim {sim}')
        mean_cen = meanCen_np[tetangga, item] 
        # print(f"mean cen {mean_cen}")
        # cek tetangga dan item pada mean cen
        # print(f"mean cen {meanCen_np[tetangga, item].shape}")
    else:  # item-based
        sim = similarity_np[tetangga, item] # ambil similarity item-item
        # print(f"sim item {sim.shape}") 
        mean_cen = meanCen_np[user, tetangga] # ambil mean centered rating

    # vektor ((a1 * b1) + (a2 * b2))
    pembilang = np.dot(mean_cen, sim)
    # print(f"pembilang {pembilang}")
    penyebut = np.sum(np.abs(sim))
    # print(f"penyebut {penyebut}")

    prediksi = mean_value + (pembilang / penyebut) if penyebut != 0 else mean_value
    # print(f"prediksi {mean_value} + {pembilang} / {penyebut}")
    return float(prediksi)

In [13]:
def prediksi_semua_matriks2(RatingMatriks, similarityFunction, mean, meanCen, k=2, jenis="user-based"):
    """
    Mengisi semua rating kosong dalam matriks dengan hasil prediksi CF.
    """
    # salin rating matriks
    hasil_matriks = RatingMatriks.copy()
    # mendapatkan ukuran matriks rating/ ambil
    num_users, num_items = hasil_matriks.shape
    # loop setiap user
    for user in range(num_users):
        # loop setiap item
        for item in range(num_items):
            if hasil_matriks.iloc[user, item] == 0:
                pred_rating = PrediksiCF(
                    RatingMatriks=RatingMatriks,
                    similarityFunction=similarityFunction,
                    mean=mean,
                    meanCen=meanCen,
                    user=user,
                    item=item,
                    k=k,
                    jenis=jenis # user-based atau item-based
                )
                # isi prediksi hasil ke dalam matriks
                hasil_matriks.iloc[user, item] = pred_rating
    return hasil_matriks

In [14]:
def simpan_semua_prediksi_k(
    RatingMatriks, similarityFunction, mean, meanCen,
    variasi_k, jenis="user-based",
    folder_output="case/prediksi/Jac/ub/5"
):
    for k in variasi_k:
        # Hitung prediksi
        prediksi = prediksi_semua_matriks2(
            RatingMatriks=RatingMatriks,
            similarityFunction=similarityFunction,
            mean=mean,
            meanCen=meanCen,
            k=k,
            jenis=jenis
        )
        # Siapkan path file
        path_file = os.path.join(folder_output, f"{k}.joblib")
        os.makedirs(folder_output, exist_ok=True)

        # Simpan hasil prediksi
        joblib.dump(prediksi, path_file)
        print(f"✔️ Prediksi untuk k={k} disimpan di: {path_file}")

In [40]:
# loop prediksi item-based
simpan_semua_prediksi_k(
    RatingMatriks=ratingMatriks,
    similarityFunction=calSimUser,
    mean=calMeanUser,
    meanCen=calMeanCenUser,
    variasi_k=[5, 10, 15, 18, 20, 25, 30, 40, 50, 100, 200],
    jenis="user-based",
    folder_output="case/prediksiModel/RJ2/ub/5"
)

✔️ Prediksi untuk k=5 disimpan di: case/prediksiModel/RJ2/ub/5\5.joblib
✔️ Prediksi untuk k=10 disimpan di: case/prediksiModel/RJ2/ub/5\10.joblib
✔️ Prediksi untuk k=15 disimpan di: case/prediksiModel/RJ2/ub/5\15.joblib
✔️ Prediksi untuk k=18 disimpan di: case/prediksiModel/RJ2/ub/5\18.joblib
✔️ Prediksi untuk k=20 disimpan di: case/prediksiModel/RJ2/ub/5\20.joblib
✔️ Prediksi untuk k=25 disimpan di: case/prediksiModel/RJ2/ub/5\25.joblib
✔️ Prediksi untuk k=30 disimpan di: case/prediksiModel/RJ2/ub/5\30.joblib
✔️ Prediksi untuk k=40 disimpan di: case/prediksiModel/RJ2/ub/5\40.joblib
✔️ Prediksi untuk k=50 disimpan di: case/prediksiModel/RJ2/ub/5\50.joblib
✔️ Prediksi untuk k=100 disimpan di: case/prediksiModel/RJ2/ub/5\100.joblib
✔️ Prediksi untuk k=200 disimpan di: case/prediksiModel/RJ2/ub/5\200.joblib
